In [1]:
"""
LNG Cargo Optimization — Multi-Agent Workflow
==============================================
Energy Commodity Trading Commercial Desk 

Scenario: Trader has a free LNG cargo out of Qatar on the 18th.
System finds the best destination by margin.

Agents:
  - intent_agent        → parses the trader's query
  - terminal_agent      → checks open slots at destinations
  - spread_agent        → calculates spark spreads & netbacks at each terminal
  - shipping_agent      → estimates freight cost per route
  - ranking_agent       → ranks destinations by net margin
  - recommendation_agent→ produces trader-friendly output
  - evaluator_agent     → scores the recommendation (offline eval)

Evaluation:
  - Routing accuracy    → did the right agents get called?
  - Recommendation quality → scored by LLM judge
  - Ground truth feedback → after cargo closes, compare recommended vs actual margin
"""

import json
import re
import anthropic
from datetime import datetime

with open("config.json", "r") as f:
    config = json.load(f)

API_KEY = config["API_KEY"]
MODEL = config["MODEL"]
client  = anthropic.Anthropic(api_key=API_KEY)


# ── Mock data (production: live feeds from Spark, Bloomberg, S&P Platts) ──────

TERMINALS = {
    "Gate LNG (Rotterdam)": {
        "region":           "Europe",
        "slots_available":  ["2024-01-18", "2024-01-20"],
        "regasification_cost_per_mmbtu": 0.35,
        "existing_vitol_obligations": 0,
        "ttf_price_per_mmbtu":        12.80,
    },
    "South Hook (UK)": {
        "region":           "Europe",
        "slots_available":  ["2024-01-19"],
        "regasification_cost_per_mmbtu": 0.38,
        "existing_vitol_obligations": 1,   # already have cargo booked
        "ttf_price_per_mmbtu":        12.65,
    },
    "Fos Cavaou (France)": {
        "region":           "Europe",
        "slots_available":  ["2024-01-18", "2024-01-21"],
        "regasification_cost_per_mmbtu": 0.40,
        "existing_vitol_obligations": 0,
        "peg_price_per_mmbtu":        12.90,
    },
    "Sodegaura (Japan)": {
        "region":           "Asia",
        "slots_available":  ["2024-01-22"],
        "regasification_cost_per_mmbtu": 0.28,
        "existing_vitol_obligations": 0,
        "jkm_price_per_mmbtu":        16.40,
    },
    "Tianjin (China)": {
        "region":           "Asia",
        "slots_available":  ["2024-01-20", "2024-01-23"],
        "regasification_cost_per_mmbtu": 0.30,
        "existing_vitol_obligations": 0,
        "jkm_price_per_mmbtu":        15.90,
    },
    "Dahej (India)": {
        "region":           "Asia",
        "slots_available":  ["2024-01-19", "2024-01-21"],
        "regasification_cost_per_mmbtu": 0.25,
        "existing_vitol_obligations": 0,
        "jkm_price_per_mmbtu":        14.20,
    },
}

SHIPPING = {
    # freight cost per MMBtu from Qatar
    "Gate LNG (Rotterdam)":  0.85,
    "South Hook (UK)":       0.90,
    "Fos Cavaou (France)":   0.88,
    "Sodegaura (Japan)":     0.55,
    "Tianjin (China)":       0.50,
    "Dahej (India)":         0.40,
}

CARGO = {
    "origin":           "RasGas T3 (Qatar)",
    "volume_mmbtu":     3_400_000,       # standard Q-Flex cargo
    "lng_cost_per_mmbtu": 8.20,          # purchase price incl. liquefaction
    "cargo_date":       "2024-01-18",
}

MARKET_CONTEXT = """
Current market snapshot (2024-01-15):
- JKM (Asian LNG benchmark): $16.40/MMBtu — premium to Europe, cold snap in NE Asia
- TTF (European gas): $12.80/MMBtu — mild winter weighing on prices
- European storage: 78% full — less urgency to restock
- Asia demand: strong, South Korea and Japan buying spot aggressively
- Panama Canal: slight delays (~1.5 days) affecting Asia routing
- Shipping market: Q-Flex rates firm at $85k/day on Middle East-Asia routes
"""


# ── Shared helpers ────────────────────────────────────────────────────────────

def call_claude(system, user_message, tools=None):
    kwargs = {
        "model":      MODEL,
        "max_tokens": 1500,
        "system":     system,
        "messages":   [{"role": "user", "content": user_message}],
    }
    if tools:
        kwargs["tools"] = tools
    return client.messages.create(**kwargs)

def get_text(response):
    return "\n".join(b.text for b in response.content if b.type == "text")

def get_tool_use(response):
    return next((b for b in response.content if b.type == "tool_use"), None)

def parse_json(raw):
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    return json.loads(match.group()) if match else {}

def parse_json_list(raw):
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    return json.loads(match.group()) if match else []

def print_agent(name, content):
    print(f"\n{'─'*65}")
    print(f"  🤖 {name}")
    print(f"{'─'*65}")
    print(content)


# ── Tools ─────────────────────────────────────────────────────────────────────

NETBACK_TOOL = {
    "name": "calculate_netback",
    "description": "Calculate net margin (netback) for delivering a cargo to a specific terminal.",
    "input_schema": {
        "type": "object",
        "properties": {
            "terminal": {
                "type": "string",
                "description": "Terminal name e.g. 'Gate LNG (Rotterdam)'"
            }
        },
        "required": ["terminal"]
    }
}

def run_netback(terminal: str) -> dict:
    """
    Netback = sale price - LNG cost - shipping - regasification
    This is the core margin calculation for each destination.
    """
    t = TERMINALS.get(terminal)
    if not t:
        return {"error": f"Terminal not found: {terminal}"}

    freight     = SHIPPING.get(terminal, 0)
    regasification = t["regasification_cost_per_mmbtu"]
    lng_cost    = CARGO["lng_cost_per_mmbtu"]

    # pick the right benchmark price
    if "ttf_price_per_mmbtu" in t:
        sale_price  = t["ttf_price_per_mmbtu"]
        benchmark   = "TTF"
    elif "peg_price_per_mmbtu" in t:
        sale_price  = t["peg_price_per_mmbtu"]
        benchmark   = "PEG"
    else:
        sale_price  = t["jkm_price_per_mmbtu"]
        benchmark   = "JKM"

    netback_per_mmbtu = sale_price - lng_cost - freight - regasification
    total_margin      = netback_per_mmbtu * CARGO["volume_mmbtu"]

    slot_available = CARGO["cargo_date"] in t["slots_available"] or \
                     any(s >= CARGO["cargo_date"] for s in t["slots_available"])

    return {
        "terminal":             terminal,
        "region":               t["region"],
        "benchmark":            benchmark,
        "sale_price":           sale_price,
        "lng_cost":             lng_cost,
        "freight":              freight,
        "regasification":       regasification,
        "netback_per_mmbtu":    round(netback_per_mmbtu, 3),
        "total_margin_usd":     round(total_margin, 0),
        "slot_available":       slot_available,
        "obligations_conflict": t["existing_vitol_obligations"] > 0,
    }


# ── Specialist Agents ─────────────────────────────────────────────────────────

def intent_agent(query: str) -> dict:
    """Parse what the trader is asking."""
    response = call_claude(
        system="""You are an intent classifier for an LNG trading desk.
Extract from the trader's query:
- cargo_origin: where the LNG is coming from
- cargo_date: the laycan / availability date (YYYY-MM-DD if possible)
- query_type: one of [cargo_optimization, market_check, position_check]
- regions_of_interest: list of regions mentioned, or ["all"] if none specified
- urgency: high / medium / low

Return JSON only: {
  "cargo_origin": "...",
  "cargo_date": "...",
  "query_type": "...",
  "regions_of_interest": [...],
  "urgency": "...",
  "summary": "one line of what trader wants"
}""",
        user_message=query
    )
    result = parse_json(get_text(response))
    print_agent("Intent Agent", json.dumps(result, indent=2))
    return result


def terminal_agent(regions: list) -> dict:
    """Check which terminals have open slots around the cargo date."""
    available = {}
    for name, data in TERMINALS.items():
        if "all" in regions or data["region"].lower() in [r.lower() for r in regions]:
            slots = [s for s in data["slots_available"] if s >= CARGO["cargo_date"]]
            if slots:
                available[name] = slots

    output = "\n".join(
        f"  ✓ {name}: slots {', '.join(slots)}"
        for name, slots in available.items()
    ) or "  No available slots found."

    print_agent("Terminal Agent", output)
    return available


def spread_agent(available_terminals: dict) -> list:
    """Calculate netback for each available terminal using the tool."""
    results = []

    for terminal in available_terminals:
        netback = run_netback(terminal)
        results.append(netback)

    # Sort by netback margin descending
    results.sort(key=lambda x: x.get("netback_per_mmbtu", -999), reverse=True)

    output = "\n".join(
        f"  {r['terminal']}: ${r['netback_per_mmbtu']:.3f}/MMBtu "
        f"(total: ${r['total_margin_usd']:,.0f}) [{r['benchmark']}]"
        for r in results
    )
    print_agent("Spread Agent", output)
    return results


def shipping_agent(netbacks: list) -> list:
    """
    Flag any shipping constraints and adjust for route risks.
    In production: live vessel positions, port congestion, Panama Canal delays.
    """
    response = call_claude(
        system="""You are a shipping analyst for an LNG trading desk.
Review these destination options and flag any shipping risks or constraints.
Consider: route length, canal access, seasonal weather, port congestion.
Be brief — one line per terminal. Return the terminal names with a risk flag: low/medium/high.
Return JSON array only: [{"terminal": "...", "shipping_risk": "low/medium/high", "note": "..."}, ...]""",
        user_message=f"""Cargo origin: {CARGO['origin']}
Cargo date: {CARGO['cargo_date']}

Market context:
{MARKET_CONTEXT}

Destination options:
{json.dumps([n['terminal'] for n in netbacks], indent=2)}"""
    )
    risks = parse_json_list(get_text(response))

    # Merge shipping risk into netback data
    risk_map = {r["terminal"]: r for r in risks}
    for nb in netbacks:
        risk_info = risk_map.get(nb["terminal"], {})
        nb["shipping_risk"] = risk_info.get("shipping_risk", "low")
        nb["shipping_note"] = risk_info.get("note", "")

    output = "\n".join(
        f"  {r['terminal']}: {r['shipping_risk'].upper()} risk — {r['shipping_note']}"
        for r in netbacks
    )
    print_agent("Shipping Agent", output)
    return netbacks


def ranking_agent(netbacks: list) -> list:
    """Rank destinations combining margin, shipping risk, and obligations."""
    response = call_claude(
        system="""You are a cargo optimization analyst at a major LNG trading house.
Rank these terminal options for a cargo deployment.

Scoring criteria:
1. Net margin per MMBtu (most important)
2. Shipping risk (high risk = penalty)
3. Existing obligations conflict (deprioritize if Vitol already has cargo there)
4. Slot timing vs cargo date (closer = better)

Return the top 3 terminals ranked by overall attractiveness.
Return JSON array only: [
  {"rank": 1, "terminal": "...", "rationale": "one line why", "action": "SEND"},
  {"rank": 2, "terminal": "...", "rationale": "one line why", "action": "CONSIDER"},
  {"rank": 3, "terminal": "...", "rationale": "one line why", "action": "BACKUP"}
]""",
        user_message=json.dumps(netbacks, indent=2)
    )
    ranked = parse_json_list(get_text(response))
    output = "\n".join(
        f"  #{r['rank']} {r['terminal']} → {r['action']}: {r['rationale']}"
        for r in ranked
    )
    print_agent("Ranking Agent", output)
    return ranked


def recommendation_agent(query: str, ranked: list, netbacks: list) -> str:
    """Produce a trader-friendly recommendation."""
    # Build netback lookup for context
    nb_map = {n["terminal"]: n for n in netbacks}

    top = ranked[0] if ranked else {}
    top_nb = nb_map.get(top.get("terminal", ""), {})

    response = call_claude(
        system="""You are a senior LNG analyst briefing a trader.
Be direct and specific. Format exactly like this:

RECOMMENDATION: [terminal name]
MARGIN: $X.XX/MMBtu | Total: $X,XXX,XXX

TOP 3 OPTIONS:
1. [terminal] — $X.XX/MMBtu — [one line reason]
2. [terminal] — $X.XX/MMBtu — [one line reason]
3. [terminal] — $X.XX/MMBtu — [one line reason]

KEY CALL:
[2-3 lines max — why top pick wins, main risk to watch]

NEXT STEP:
[One specific action the trader should take now]

No waffle. The trader needs to act in the next 30 minutes.""",
        user_message=f"""Trader query: {query}

Cargo: {CARGO['volume_mmbtu']:,} MMBtu from {CARGO['origin']}, laycan {CARGO['cargo_date']}

Ranked options:
{json.dumps(ranked, indent=2)}

Netback details:
{json.dumps(netbacks, indent=2)}

Market context:
{MARKET_CONTEXT}"""
    )
    result = get_text(response)
    print_agent("Recommendation Agent", result)
    return result


def evaluator_agent(query: str, recommendation: str, netbacks: list) -> dict:
    """Score the recommendation quality — used for offline model improvement."""
    response = call_claude(
        system="""You are a senior LNG trading desk evaluator.
Score this AI-generated cargo recommendation on:
- margin_accuracy: did it correctly identify the highest-margin destination? (1-5)
- risk_awareness: did it flag shipping and obligation risks? (1-5)
- actionability: can the trader act on this immediately? (1-5)
- clarity: would a busy trader understand this in 10 seconds? (1-5)
- overall: overall recommendation quality (1-5)

Return JSON only: {
  "margin_accuracy": N,
  "risk_awareness": N,
  "actionability": N,
  "clarity": N,
  "overall": N,
  "top_destination_correct": true/false,
  "improvement": "one specific thing that would make this better"
}""",
        user_message=f"""Trader query: {query}

Recommendation:
{recommendation}

Ground truth netbacks (for scoring margin accuracy):
{json.dumps(netbacks, indent=2)}"""
    )
    result = parse_json(get_text(response))
    print_agent("Evaluator Agent", json.dumps(result, indent=2))
    return result


# ── Dynamic Router ────────────────────────────────────────────────────────────

def route(intent: dict) -> list:
    """
    Build pipeline from intent — pure Python, no extra Claude call.
    Fast and predictable for a latency-sensitive trading environment.
    """
    pipeline = ["terminal", "spread", "shipping", "ranking", "recommendation", "evaluator"]

    # If trader only wants a market check, skip the full optimization
    if intent.get("query_type") == "market_check":
        pipeline = ["recommendation", "evaluator"]

    return pipeline


# ── Orchestrator ──────────────────────────────────────────────────────────────

def orchestrator(trader_query: str) -> dict:
    print(f"\n{'═'*65}")
    print(f"  TRADER QUERY: {trader_query}")
    print(f"  TIME: {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"{'═'*65}")

    context = {}

    # Step 1: Always classify intent
    intent = intent_agent(trader_query)
    context["intent"] = intent
    regions = intent.get("regions_of_interest", ["all"])

    # Step 2: Route
    pipeline = route(intent)
    print(f"\n  📋 Pipeline: intent → {' → '.join(pipeline)}")

    # Step 3: Run agents in sequence, passing context forward
    for agent in pipeline:

        if agent == "terminal":
            context["available_terminals"] = terminal_agent(regions)

        elif agent == "spread":
            context["netbacks"] = spread_agent(context["available_terminals"])

        elif agent == "shipping":
            context["netbacks"] = shipping_agent(context["netbacks"])

        elif agent == "ranking":
            context["ranked"] = ranking_agent(context["netbacks"])

        elif agent == "recommendation":
            context["recommendation"] = recommendation_agent(
                trader_query,
                context.get("ranked", []),
                context.get("netbacks", [])
            )

        elif agent == "evaluator":
            context["eval"] = evaluator_agent(
                trader_query,
                context["recommendation"],
                context.get("netbacks", [])
            )

    print(f"\n{'═'*65}")
    print(f"  DONE — Eval score: {context['eval'].get('overall')}/5")
    print(f"{'═'*65}\n")

    return context


# ── Ground truth eval (runs weeks later when cargo closes) ────────────────────

def ground_truth_eval(recommendation_log: dict, actual_terminal: str, actual_margin: float):
    """
    After the cargo closes, compare what we recommended vs what actually happened.
    This is your long-term model improvement signal.
    In production: store recommendation_log in a database, run this when trade settles.
    """
    netbacks        = recommendation_log.get("netbacks", [])
    ranked          = recommendation_log.get("ranked", [])
    recommended     = ranked[0]["terminal"] if ranked else "unknown"
    recommended_nb  = next((n for n in netbacks if n["terminal"] == recommended), {})
    predicted_margin = recommended_nb.get("netback_per_mmbtu", 0)

    correct = recommended == actual_terminal
    margin_error = abs(predicted_margin - actual_margin)

    print(f"\n{'═'*65}")
    print(f"  GROUND TRUTH EVAL (post-trade settlement)")
    print(f"{'─'*65}")
    print(f"  Recommended: {recommended} @ ${predicted_margin:.3f}/MMBtu")
    print(f"  Actual:      {actual_terminal} @ ${actual_margin:.3f}/MMBtu")
    print(f"  Routing correct: {correct}")
    print(f"  Margin prediction error: ${margin_error:.3f}/MMBtu")
    print(f"{'═'*65}\n")

    return {
        "recommended":      recommended,
        "actual":           actual_terminal,
        "routing_correct":  correct,
        "margin_error":     margin_error,
        "predicted_margin": predicted_margin,
        "actual_margin":    actual_margin,
    }


# ── Offline eval suite ────────────────────────────────────────────────────────

def run_eval_suite():
    """Run across multiple query types and summarize scores."""
    test_cases = [
        {
            "query": "I have a free cargo out of Qatar on the 18th, where should it go?",
            "expected_type": "cargo_optimization",
        },
        {
            "query": "Free Qatar cargo Jan 18 — Asia or Europe better right now?",
            "expected_type": "cargo_optimization",
        },
        {
            "query": "What's the JKM-TTF spread looking like today?",
            "expected_type": "market_check",
        },
    ]

    results = []

    for i, tc in enumerate(test_cases, 1):
        print(f"\n{'#'*65}")
        print(f"  TEST {i}/{len(test_cases)}: {tc['query']}")
        print(f"{'#'*65}")

        output      = orchestrator(tc["query"])
        intent      = output.get("intent", {})
        eval_scores = output.get("eval", {})

        results.append({
            "query":            tc["query"],
            "routing_correct":  intent.get("query_type") == tc["expected_type"],
            "eval":             eval_scores,
        })

    # Summary
    print(f"\n{'═'*65}")
    print("  EVAL SUITE SUMMARY")
    print(f"{'─'*65}")

    routing_acc = sum(r["routing_correct"] for r in results) / len(results) * 100
    scores      = [r["eval"].get("overall", 0) for r in results]
    avg_score   = sum(scores) / len(scores)

    print(f"  Routing accuracy:  {routing_acc:.0f}%")
    print(f"  Avg overall score: {avg_score:.1f}/5")
    print(f"  Avg actionability: {sum(r['eval'].get('actionability',0) for r in results)/len(results):.1f}/5")
    print(f"  Avg clarity:       {sum(r['eval'].get('clarity',0) for r in results)/len(results):.1f}/5")
    print(f"\n  Improvements flagged:")
    for r in results:
        imp = r["eval"].get("improvement")
        if imp:
            print(f"  • {imp}")
    print(f"{'═'*65}\n")

    return results


# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # Single query
    result = orchestrator(
        "I have a free cargo out of Qatar on the 18th, where should it go?"
    )

    # Simulate ground truth eval (runs after cargo settles, e.g. 2 weeks later)
    ground_truth_eval(
        recommendation_log=result,
        actual_terminal="Sodegaura (Japan)",   # where it actually went
        actual_margin=7.42                      # actual realised margin/MMBtu
    )

    # Full eval suite across query types
    # run_eval_suite()


═════════════════════════════════════════════════════════════════
  TRADER QUERY: I have a free cargo out of Qatar on the 18th, where should it go?
  TIME: 2026-06-15 01:23 UTC
═════════════════════════════════════════════════════════════════

─────────────────────────────────────────────────────────────────
  🤖 Intent Agent
─────────────────────────────────────────────────────────────────
{
  "cargo_origin": "Qatar",
  "cargo_date": "2025-01-18",
  "query_type": "cargo_optimization",
  "regions_of_interest": [
    "all"
  ],
  "urgency": "medium",
  "summary": "Trader seeks optimal destination for a free Qatari cargo with an 18th laycan."
}

  📋 Pipeline: intent → terminal → spread → shipping → ranking → recommendation → evaluator

─────────────────────────────────────────────────────────────────
  🤖 Terminal Agent
─────────────────────────────────────────────────────────────────
  ✓ Gate LNG (Rotterdam): slots 2024-01-18, 2024-01-20
  ✓ South Hook (UK): slots 2024-01-19
  ✓ Fos Cava